# Phase 1 — Exploratory Data Analysis (EDA)
**Project:** Deep Loan Default Prediction via VAE-based Anomaly Detection  
**Dataset:** Lending Club Accepted Loans (2007–2018)  

**Goals:**
1. Understand raw data shape, dtypes, and missing values
2. Examine class distribution (Fully Paid vs. Charged Off)
3. Identify key application-time features
4. Spot data quality issues before preprocessing


## 0. Imports & Config

In [2]:
import sys, os
sys.path.append(os.path.join('..'))  # make src/ importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from src.config import (
    RAW_CSV, TARGET_COL, NORMAL_LABEL, ANOMALY_LABEL,
    APPLICATION_FEATURES, FIGURES_DIR
)

os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Imports OK')
print(f'   Looking for data at: {RAW_CSV}')

Imports OK
   Looking for data at: d:\project\Deep-Loan-Default-VAE\data\raw\accepted_2007_to_2018Q4.csv


## 1. Load Raw Data

In [3]:
# The Lending Club CSV is large (~2 GB). We load with low_memory=False
# to let pandas infer dtypes across the whole file.
df_raw = pd.read_csv(RAW_CSV, low_memory=False)

print(f'Raw shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

ParserError: Error tokenizing data. C error: out of memory

## 2. Dataset Overview

In [ ]:
# Data types summary
dtype_counts = df_raw.dtypes.value_counts()
print('Column dtype breakdown:')
print(dtype_counts)
print()

# Memory usage
mem_mb = df_raw.memory_usage(deep=True).sum() / 1e6
print(f'Memory usage: {mem_mb:.1f} MB')

## 3. Target Column — Class Distribution

In [ ]:
# Full value counts of loan_status
print('All loan_status values:')
print(df_raw[TARGET_COL].value_counts())

In [ ]:
# Filter to completed loans only (our working set)
df = df_raw[df_raw[TARGET_COL].isin([NORMAL_LABEL, ANOMALY_LABEL])].copy()
print(f'Completed loans (Fully Paid + Charged Off): {len(df):,} rows')
print()

counts = df[TARGET_COL].value_counts()
pct    = df[TARGET_COL].value_counts(normalize=True) * 100

summary = pd.DataFrame({'Count': counts, 'Percent (%)': pct.round(2)})
print(summary)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ['#4f46e5', '#dc2626']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Counts', fontweight='bold')
axes[0].set_ylabel('Number of Loans')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.01,
                 f'{val:,}', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportions', fontweight='bold')

plt.suptitle('Loan Status Distribution (Completed Loans Only)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '01_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/01_class_distribution.png')

## 4. Missing Value Analysis

In [ ]:
# Missing value rate per column (only showing columns with >0% missing)
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing = missing[missing > 0]
print(f'Columns with missing values: {len(missing)} / {df.shape[1]}')

# Show top 30
top_missing = missing.head(30)

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = ['#dc2626' if v > 50 else '#d97706' if v > 20 else '#4f46e5' for v in top_missing.values]
bars = ax.barh(top_missing.index[::-1], top_missing.values[::-1], color=colors_bar[::-1], edgecolor='white')
ax.set_xlabel('Missing (%)')
ax.set_title('Top 30 Columns by Missing Rate', fontweight='bold')
ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='>50% missing')
ax.axvline(20, color='orange', linestyle='--', alpha=0.5, label='>20% missing')
ax.legend()
for bar, val in zip(bars, top_missing.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '02_missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/02_missing_values.png')

## 5. Application-Time Features — Quick Look

In [ ]:
# Check which of our selected features exist in this dataset
available = [f for f in APPLICATION_FEATURES if f in df.columns]
missing_cols = [f for f in APPLICATION_FEATURES if f not in df.columns]

print(f'Available application features : {len(available)}/{len(APPLICATION_FEATURES)}')
if missing_cols:
    print(f'Not found in dataset          : {missing_cols}')

df_feats = df[available + [TARGET_COL]].copy()
df_feats.head(3)

In [ ]:
# Missing rate within our feature subset
feat_missing = (df_feats[available].isnull().sum() / len(df_feats) * 100).sort_values(ascending=False)
feat_missing = feat_missing[feat_missing > 0]
print('Missing rates in selected features:')
print(feat_missing if len(feat_missing) else 'None — all features complete ✅')

## 6. Numerical Feature Distributions (Normal vs. Default)

In [ ]:
num_feats = df_feats[available].select_dtypes(include=[np.number]).columns.tolist()
print(f'Numerical features: {num_feats}')

n_cols = 3
n_rows = (len(num_feats) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

normal_df  = df_feats[df_feats[TARGET_COL] == NORMAL_LABEL]
anomaly_df = df_feats[df_feats[TARGET_COL] == ANOMALY_LABEL]

for i, feat in enumerate(num_feats):
    ax = axes[i]
    # Use 99th percentile clip to suppress outlier distortion
    clip_max = df_feats[feat].quantile(0.99)
    
    sns.kdeplot(normal_df[feat].clip(upper=clip_max).dropna(),
                ax=ax, label='Fully Paid', color='#4f46e5', fill=True, alpha=0.3)
    sns.kdeplot(anomaly_df[feat].clip(upper=clip_max).dropna(),
                ax=ax, label='Charged Off', color='#dc2626', fill=True, alpha=0.3)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=7)

# Hide empty axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions: Normal vs. Default', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/03_feature_distributions.png')

## 7. Correlation Heatmap (Numerical Features)

In [ ]:
corr_matrix = df_feats[num_feats].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 7}, ax=ax
)
ax.set_title('Correlation Heatmap — Application-Time Numerical Features', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '04_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/04_correlation_heatmap.png')

## 8. Categorical Features — Top Categories

In [ ]:
cat_feats = df_feats[available].select_dtypes(include=['object']).columns.tolist()
print(f'Categorical features: {cat_feats}')

fig, axes = plt.subplots(1, len(cat_feats), figsize=(5 * len(cat_feats), 4))
if len(cat_feats) == 1:
    axes = [axes]

for ax, feat in zip(axes, cat_feats):
    top_cats = df_feats[feat].value_counts().head(8)
    ax.barh(top_cats.index[::-1], top_cats.values[::-1], color='#4f46e5', edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Count')

plt.suptitle('Top Categories per Categorical Feature', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '05_categorical_features.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/05_categorical_features.png')

## 9. Default Rate by Grade

In [ ]:
if 'grade' in df_feats.columns:
    grade_default = (
        df_feats.groupby('grade')[TARGET_COL]
        .apply(lambda x: (x == ANOMALY_LABEL).sum() / len(x) * 100)
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    colors_grade = ['#16a34a' if v < 10 else '#d97706' if v < 20 else '#dc2626' for v in grade_default.values]
    bars = ax.bar(grade_default.index, grade_default.values, color=colors_grade, edgecolor='white', width=0.6)
    for bar, val in zip(bars, grade_default.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', fontsize=9)
    ax.set_xlabel('Loan Grade')
    ax.set_ylabel('Default Rate (%)')
    ax.set_title('Default Rate by Loan Grade', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, '06_default_by_grade.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → reports/figures/06_default_by_grade.png')
else:
    print('grade column not found, skipping.')

## 10. EDA Summary & Notes for Phase 2

In [ ]:
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print(f'Total completed loans    : {len(df):,}')
print(f'Fully Paid (Normal)      : {(df[TARGET_COL]==NORMAL_LABEL).sum():,} ({(df[TARGET_COL]==NORMAL_LABEL).mean()*100:.1f}%)')
print(f'Charged Off (Anomaly)    : {(df[TARGET_COL]==ANOMALY_LABEL).sum():,} ({(df[TARGET_COL]==ANOMALY_LABEL).mean()*100:.1f}%)')
print()
print(f'Application features     : {len(available)} available, {len(missing_cols)} not found')
print(f'Numerical features       : {len(num_feats)}')
print(f'Categorical features     : {len(cat_feats)}')
print()
print('Next Step → Phase 2: Data Preprocessing Pipeline')
print('  notebooks/02_preprocessing.ipynb')
print('=' * 60)